In [ ]:
import pp_heig_plot as pp_plot
import pp_heig_simulation as pp_sim
from datetime import time
from pp_importer import PandaPowerImporter
import pandapower as pp
import pandapower.shortcircuit as sc
import pandapower.plotting.plotly as pplotly


In [ ]:
net_file_path = "experiment/3_bus_example.xlsx"
net = PandaPowerImporter().read_excel(file_path=net_file_path)
net

We can plot a simplified diagram of our network using the following function:

- By adding a filename, the plot will be saved in a png format in the default folder _plot_.
- We can change the folder name using the folder parameter.
- We can view the equipment parameters in the plot by moving the mouse over them.
- The network is well traced when it is tree-like. In the case of a mesh grid, a coordinate parameter must be added to the buses.

In [ ]:
pp_plot.plot_power_network(
    net=net, plot_title="3-bus example", filename="3_bus_example"
)



We can run a simple power flow and visualise result using the following functions:

In [ ]:
pp.runpp(net)
pp_plot.plot_powerflow_result(
    net=net, plot_title="3-bus powerflow results", filename="3_bus_pp_result"
)
net.res_bus

## Timeseries powerflow simulation


We can create power profiles from excel files to perform timeseries powerflow simulations. After having been loaded, the resulting object is a dictionary of dataframe:

- Keys is the equipment name where profile are related to.
- Values can be active and reactive power profile table.

In [ ]:
profile_file_path = "experiment/3_bus_power_profile.xlsx"
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)
print(time_series.keys())
print(time_series["load"].keys())
time_series["load"]["p_mw"]

In this example, the file loaded contains two different profiles for loads. If we take a look in the load **panda**power table we can see that the `profile_mapping` parameter of the load is set to 0. It means that power profiles applied to this load will be the 0.

In [ ]:
pp_sim.apply_power_profile(
    net=net, equipment="load", power_profiles=time_series["load"]
)

Then we need to create an output writer which will store simulation results:

- Default results stored are `res_bus.vm_pu`, `res_line.loading_percent`, `res_trafo.loading_percent`.
- We can add other results using `add_results` parameters.

In [ ]:
pp_sim.create_output_writer(net=net, add_results=["res_line.p_from_mw"])

Finally, we can run times series simulation and plot results – as follows:

In [ ]:
result_df = pp_sim.run_time_simulation(net=net)
print()
pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage",
    filename="voltage_result",
)
print()
pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="line power",
    filename="line_result",
)
print()
pp_plot.plot_timestamps_powerflow_result(
    net=net, filename="net_result_12h", plot_time=time(hour=12)
)

We can use the second power profile loaded for the excel file. To do this, we just need to modify the `profile_mapping` parameter before applying once again the power profile:

In [ ]:
net.load.loc[0, "profile_mapping"] = 1
pp_sim.apply_power_profile(
    net=net, equipment="load", power_profiles=time_series["load"]
)
result_df = pp_sim.run_time_simulation(net=net)
print()
pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage",
    filename="voltage_result",
)
print()
pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="line power",
    filename="line_result",
)

We can also scale our power profiles modifying `scaling` parameters:

In [ ]:
net.load.loc[0, "scaling"] = 5
result_df = pp_sim.run_time_simulation(net=net)
pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage",
    filename="voltage_result",
)
print()
pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="line power",
    filename="line_result",
)

In [ ]:
net.trafo.T

# References

- [Pandapower 'Getting started'](http://www.pandapower.org/start/)
- [Pandapower's documentation](https://pandapower.readthedocs.io/en/v2.13.1/index.html)
- [Pandapower's tutorials on GitHub](https://github.com/e2nIEE/pandapower/tree/v2.13.1/tutorials)

## Citing pandapower

 &copy; Copyright 2016-2023 by Fraunhofer IEE and University of Kassel. Revision 2feba868.

```latex
@article{pandapower.2018,
author={L. Thurner and A. Scheidler and F. Schafer and J. H. Menke and J. Dollichon and F. Meier and S. Meinecke and M. Braun},
journal={IEEE Transactions on Power Systems},
title={pandapower - an Open Source Python Tool for Convenient Modeling, Analysis and Optimization of Electric Power Systems},
year={2018},
doi={10.1109/TPWRS.2018.2829021},
url={https://arxiv.org/abs/1709.06743},
ISSN={0885-8950}
}
```